# 💳 Credit Card Fraud Detection
## Step 3 — Evaluate & Compare Models

In [ ]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve, precision_recall_curve,
    average_precision_score
)

# Load data and models
X_train, X_test, y_train, y_test = joblib.load('data/train_test_split.pkl')
dt_model  = joblib.load('models/decision_tree.pkl')
rf_model  = joblib.load('models/random_forest.pkl')
xgb_model = joblib.load('models/xgboost.pkl')

models = {
    'Decision Tree':  dt_model,
    'Random Forest':  rf_model,
    'XGBoost':        xgb_model
}
colors = {'Decision Tree': 'royalblue', 'Random Forest': 'seagreen', 'XGBoost': 'crimson'}

print('Everything loaded!')

### 1. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, (name, model) in zip(axes, models.items()):
    preds = model.predict(X_test)
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Legit', 'Fraud'],
                yticklabels=['Legit', 'Fraud'])
    ax.set_title(f'{name}', fontsize=12, fontweight='bold')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')

plt.suptitle('Confusion Matrices', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('plots/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

### 2. ROC Curves

In [ ]:
plt.figure(figsize=(8, 6))

for name, model in models.items():
    proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=colors[name], lw=2)

plt.plot([0,1],[0,1],'k--', alpha=0.4, label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — Model Comparison', fontsize=13)
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('plots/roc_curves.png', dpi=150)
plt.show()

### 3. Precision-Recall Curves
> More informative than ROC for imbalanced datasets

In [ ]:
plt.figure(figsize=(8, 6))

for name, model in models.items():
    proba = model.predict_proba(X_test)[:, 1]
    precision, recall, _ = precision_recall_curve(y_test, proba)
    ap = average_precision_score(y_test, proba)
    plt.plot(recall, precision, label=f'{name} (AP={ap:.3f})', color=colors[name], lw=2)

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curves — Model Comparison', fontsize=13)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('plots/pr_curves.png', dpi=150)
plt.show()

### 4. Feature Importance (XGBoost)

In [ ]:
feature_names = X_test.columns if hasattr(X_test, 'columns') else [f'V{i}' for i in range(X_test.shape[1])]

importances = pd.Series(xgb_model.feature_importances_, index=feature_names)
top15 = importances.nlargest(15)

plt.figure(figsize=(9, 6))
top15.sort_values().plot(kind='barh', color='crimson', alpha=0.8)
plt.title('Top 15 Feature Importances — XGBoost', fontsize=13)
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('plots/feature_importance.png', dpi=150)
plt.show()

### 5. Final Scorecard

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

rows = []
for name, model in models.items():
    preds = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    rows.append({
        'Model':     name,
        'ROC-AUC':   round(roc_auc_score(y_test, proba), 4),
        'Avg Prec':  round(average_precision_score(y_test, proba), 4),
        'F1 (Fraud)':round(f1_score(y_test, preds), 4),
        'Precision': round(precision_score(y_test, preds), 4),
        'Recall':    round(recall_score(y_test, preds), 4),
    })

scorecard = pd.DataFrame(rows).set_index('Model')
print('=== FINAL SCORECARD ===')
print(scorecard.to_string())
print('\n💡 Focus on Recall (catching all fraud) and F1 for imbalanced data.')